# 01 · Chuẩn bị VNTS + train detector YOLO11n (1 lớp và 52 lớp)

Kaggle **GPU T4**, bật Internet. Input:
- dataset `maitam/vietnamese-traffic-signs` (VNTS, 52 lớp đặt tên theo mã QCVN, CC BY-SA 4.0)
- dataset `vn-dashcam-vision-code` (code repo này)

Train 2 detector để so sánh:
- `1cls`: mọi biển gộp 1 lớp "sign" -> tầng 1 của pipeline 2 tầng
- `ncls`: 52 lớp -> baseline 1 tầng

Ultralytics là AGPL-3.0: repo public thì ổn, dùng thương mại phải mở mã hoặc mua license.

In [ ]:
import glob, json, os, sys

CODE = glob.glob("/kaggle/input/**/dashcam/__init__.py", recursive=True)[0].rsplit("/dashcam", 1)[0]
VNTS = [d for d in glob.glob("/kaggle/input/*") if "traffic" in d.lower()][0]
sys.path.insert(0, CODE)
os.environ["PYTHONPATH"] = CODE
print(CODE, VNTS)
!pip install -q ultralytics
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!python -m dashcam.prepare_vnts --src {VNTS} --out datasets/vnts
print(json.dumps(json.load(open("datasets/vnts/class_stats.json"))["train"], ensure_ascii=False)[:1500])

## Phân bố lớp (lệch rất mạnh)

In [ ]:
import matplotlib.pyplot as plt

st = json.load(open("datasets/vnts/class_stats.json"))["train"]
items = sorted(st.items(), key=lambda x: -x[1])
plt.figure(figsize=(16, 4))
plt.bar([k for k, _ in items], [v for _, v in items])
plt.xticks(rotation=90, fontsize=7); plt.yscale("log"); plt.title("số crop train mỗi lớp (log)")
plt.tight_layout(); plt.savefig("class_dist.png", dpi=110)

## Copy-paste cho lớp hiếm (< 50 mẫu train)

In [ ]:
!python -m dashcam.synth_rare --data datasets/vnts --rare 50 --per-class 150

## Train YOLO11n
imgsz 640: biển xa chỉ ~15px nên 960 sẽ tốt hơn cho recall, nhưng chạy CPU chậm ~2.2 lần.
Giữ 640 vì mục tiêu là chạy được trên laptop; ghi lại đánh đổi này trong NOTES.

In [ ]:
from ultralytics import YOLO

for name in ("1cls", "ncls"):
    m = YOLO("yolo11n.pt")
    m.train(data=f"datasets/vnts/yolo_{name}/data.yaml", imgsz=640, epochs=60, patience=15, batch=32,
            fliplr=0.0,  # KHÔNG lật ngang: cấm rẽ trái <-> cấm rẽ phải
            mosaic=1.0, degrees=5, hsv_v=0.5, project="runs", name=f"det_{name}", exist_ok=True, verbose=False)

In [ ]:
for name in ("1cls", "ncls"):
    m = YOLO(f"runs/det_{name}/weights/best.pt")
    r = m.val(data=f"datasets/vnts/yolo_{name}/data.yaml", split="test", imgsz=640, verbose=False)
    print(name, "mAP50", round(r.box.map50, 4), "mAP50-95", round(r.box.map, 4))
    m.export(format="onnx", imgsz=640, opset=17, simplify=True)

In [ ]:
!mkdir -p models && cp runs/det_1cls/weights/best.onnx models/detector_1cls.onnx && cp runs/det_ncls/weights/best.onnx models/detector_ncls.onnx
!cp datasets/vnts/names.json models/ && ls -la models
# giữ crops + class_stats cho notebook 02
!tar czf crops.tgz datasets/vnts/crops datasets/vnts/names.json datasets/vnts/class_stats.json